# [센서 실습 22] 심박 센서 – MAX30102 (PPG)

광혈류(PPG) 센서 **MAX30102**의 FIFO 데이터를 I2C로 읽어 맥파 파형을 얻고, **봉우리(peak) 검출**로 심박수(BPM)를 계산합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
심장이 뛸 때마다 손가락 혈관에 피가 밀려들어 와 **혈액량이 늘었다 줄었다** 합니다. 이를 빛으로 재는 방법이 **PPG(광혈류측정)** 입니다.
- MAX30102가 **적외선 LED** 빛을 손가락에 쏘고, 돌아오는 빛의 양을 18비트 숫자로 측정
- 혈액이 많을 때(수축기) 빛을 더 흡수 → 측정값이 주기적으로 오르내림
- 측정값 = 큰 고정 성분(**DC**: 피부·뼈·정맥) + 작은 맥동 성분(**AC**: 동맥 혈류, DC의 1~2%)
- 처리 순서: **DC 제거** → **봉우리 찾기** → 봉우리 사이 간격(초)으로 **BPM = 60 ÷ 간격**
- 스마트워치 심박 측정과 같은 원리 (단, 이 실습은 **의료용이 아님**)

### 부품 사양
| 항목 | 내용 |
|---|---|
| LED | 빨강(660nm) + 적외선(880nm) |
| ADC | 18비트, 50~3200 샘플/초 |
| FIFO | 32샘플 저장 |
| 전원 | 1.8V(칩) + 3.3V(LED), 모듈은 3.3V |
| 통신 | I2C 주소 0x57 |

## 2. 라즈베리파이 연결 방법
**준비물**: MAX30102 모듈, 점퍼선 4~5개

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| MAX30102 VCC | 3.3V (1번 핀) |  |
| MAX30102 GND | GND (6번 핀) |  |
| MAX30102 SCL | GPIO3 / SCL (5번 핀) | I2C 클록 |
| MAX30102 SDA | GPIO2 / SDA (3번 핀) | I2C 데이터 (주소 0x57) |
| MAX30102 INT | GPIO17 (11번 핀, 선택) | 데이터 준비 인터럽트 |

손가락을 **너무 세게 누르면** 혈류가 막혀 파형이 약해집니다. 가볍게 올려 두세요.

**인터페이스 활성화**
- **I2C**: `sudo raspi-config` → Interface Options → **I2C** → Enable, 연결 확인: `i2cdetect -y 1`

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install smbus2 scipy
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
FIFO 6바이트를 해석하는 함수와, 적외선 파형에서 심박수를 계산하는 함수입니다.

In [ ]:
def parse_fifo(d):
    """FIFO 6바이트 → 빨강·적외선 값 (각 18비트)"""
    red = ((d[0] << 16) | (d[1] << 8) | d[2]) & 0x3FFFF
    ir = ((d[3] << 16) | (d[4] << 8) | d[5]) & 0x3FFFF
    return {"red": red, "ir": ir}

def make_fifo(red, ir):
    """가상 센서용: 값 → FIFO 6바이트 (실제 칩과 같은 형식)"""
    red, ir = int(red) & 0x3FFFF, int(ir) & 0x3FFFF
    return [red >> 16, (red >> 8) & 0xFF, red & 0xFF, ir >> 16, (ir >> 8) & 0xFF, ir & 0xFF]

FINGER_MIN = 50000          # 적외선 값이 이보다 작으면 손가락이 없는 것으로 판단

from scipy.signal import find_peaks

def estimate_bpm(ir, fs):
    """적외선 파형 → 심박수(BPM). 손가락이 없으면 None"""
    ir = np.asarray(ir, dtype=float)
    if np.mean(ir < FINGER_MIN) > 0.05:          # 손가락이 없거나 올리는 중
        return None
    ac = -(ir - pd.Series(ir).rolling(int(fs), center=True, min_periods=1).mean().values)   # DC 제거·뒤집기
    peaks, _ = find_peaks(ac, distance=int(fs * 0.33), prominence=ac.std() * 0.8)          # 최대 180BPM
    if len(peaks) < 2:
        return None
    return round(60 / (np.diff(peaks).mean() / fs), 1)

### 4-2. 실제 센서 클래스
리셋 → SpO2 모드 → 100샘플/초 설정 후, FIFO에서 6바이트씩 읽습니다. 라이브러리 없이 **데이터시트의 레지스터를 직접** 다룹니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealMAX30102:
    """MAX30102 레지스터 직접 제어 (SpO2 모드, 100 샘플/초)"""
    ADDR = 0x57
    FIFO_WR_PTR, FIFO_RD_PTR, FIFO_DATA = 0x04, 0x06, 0x07
    FIFO_CONFIG, MODE_CONFIG, SPO2_CONFIG = 0x08, 0x09, 0x0A
    LED1_PA, LED2_PA = 0x0C, 0x0D

    def __init__(self, bus=1):
        from smbus2 import SMBus
        self.bus = SMBus(bus)
        w = lambda reg, val: self.bus.write_byte_data(self.ADDR, reg, val)
        w(self.MODE_CONFIG, 0x40); time.sleep(0.1)       # 리셋
        w(self.FIFO_CONFIG, 0x4F)                         # 4개 평균, FIFO 순환
        w(self.MODE_CONFIG, 0x03)                         # SpO2 모드 (빨강 + 적외선)
        w(self.SPO2_CONFIG, 0x27)                         # 범위 4096nA, 100sps, 18비트
        w(self.LED1_PA, 0x24); w(self.LED2_PA, 0x24)      # LED 밝기 약 7mA
        w(self.FIFO_WR_PTR, 0); w(self.FIFO_RD_PTR, 0)

    def read(self):
        d = self.bus.read_i2c_block_data(self.ADDR, self.FIFO_DATA, 6)
        return parse_fifo(d)

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 처음 2초 | 손가락을 아직 올리지 않음 | 측정 시작 상황 |
| 2 ~ 15초 | 안정 시 심박 약 70 BPM | 성인 안정 시 60~100 |
| 15 ~ 30초 | 계단을 오른 뒤 약 105 BPM으로 상승 | 운동 후 |
| 신호 크기 | DC 약 110,000, AC 약 1,600 (1.5%) | 실제 MAX30102 손가락 측정값 수준 |
| 기타 | 호흡에 따른 느린 흔들림, 잡음 | 실제 파형 특성 |

가상 센서는 PPG 파형을 만들어 **실제 칩과 같은 FIFO 6바이트**로 바꾼 뒤 해석기를 통과시킵니다.

In [ ]:
class VirtualMAX30102:
    """손가락 혈류(PPG) 파형을 만들어 실제 FIFO 형식으로 내보내는 가상 MAX30102"""
    def __init__(self, dt=0.01, seed=22):
        self.dt, self.t, self.phase = dt, 0.0, 0.0
        self.rng = np.random.default_rng(seed)

    @staticmethod
    def pulse_shape(ph):
        """심장 박동 1회 모양: 수축기 봉우리 + 작은 중복맥 (0~1 주기)"""
        return np.exp(-((ph - 0.15) / 0.07) ** 2) + 0.35 * np.exp(-((ph - 0.45) / 0.08) ** 2)

    def read(self):
        t = self.t
        hr = 70 + 35 / (1 + np.exp(-(t - 17) / 1.5))                                  # 심박수 시나리오 (BPM)
        spo2 = 98                              # 산소포화도 시나리오 (%)
        self.phase = (self.phase + hr / 60 * self.dt) % 1.0
        p = self.pulse_shape(self.phase)
        ir_dc, ir_ac = 110000, 1600                     # 손가락 반사광: 큰 DC + 작은 AC
        r = (110 - spo2) / 25                           # 빨강/적외선 흡수 비율 R
        red_dc = 90000
        red_ac = r * ir_ac / ir_dc * red_dc
        drift = 800 * np.sin(2 * np.pi * 0.2 * t)       # 호흡에 따른 느린 흔들림
        ir = ir_dc + drift - ir_ac * p + self.rng.normal(0, 60)
        red = red_dc + drift * 0.8 - red_ac * p + self.rng.normal(0, 60)
        if t < 2:
            ir, red = self.rng.normal(3000, 300), self.rng.normal(2500, 300)   # 손가락 없음
        self.t += self.dt
        return parse_fifo(make_fifo(red, ir))

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualMAX30102(dt=0.01)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
100Hz(0.01초 간격)로 30초 동안 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 3000               # 측정 횟수
INTERVAL = 0.01               # 측정 간격(초)

sensor = RealMAX30102() if USE_REAL_SENSOR else VirtualMAX30102(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
5초 구간마다 심박수를 계산합니다. **60 미만 서맥, 100 초과 빈맥**으로 표시합니다. (교육용 판정)

In [ ]:
FS = 1 / INTERVAL
rows = []
for s in range(0, int(N_SAMPLES * INTERVAL), 5):
    seg = df[(df.time_s >= s) & (df.time_s < s + 5)]
    bpm = estimate_bpm(seg["ir"].values, FS)
    state = "손가락 없음" if bpm is None else "서맥" if bpm < 60 else "빈맥" if bpm > 100 else "정상"
    rows.append({"구간(초)": f"{s}~{s + 5}", "BPM": bpm, "판정": state})
hr = pd.DataFrame(rows)
print(hr.to_string(index=False))

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6))
seg = df[(df.time_s >= 5) & (df.time_s < 10)]
ax[0].plot(seg.time_s, seg.ir, "k-", lw=1); ax[0].set_title("MAX30102 적외선 맥파 (5~10초 확대)")
ax[0].set_ylabel("IR 값"); ax[0].set_xlabel("시간 (초)"); ax[0].grid(alpha=0.3)
x = [int(r.split("~")[0]) + 2.5 for r in hr["구간(초)"]]
ax[1].plot(x, hr.BPM, "k-o"); ax[1].axhspan(60, 100, color="0.92", label="정상 60~100")
ax[1].set_ylabel("심박수 (BPM)"); ax[1].set_xlabel("시간 (초)"); ax[1].set_ylim(40, 130)
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "맥박(심박)센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "MAX30102 (PPG)",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/heart_rate", payload)
# client.disconnect()

## 직접 해 보기
- 구간 길이를 5초에서 10초로 바꾸면 심박수 변화가 어떻게 보이나요?
- 잡음을 키우면(`normal(0, 60)` → 400) 봉우리 검출이 어떻게 되나요?
- 봉우리 간격의 변화(심박 변이도, HRV)를 계산해 보세요.